# M3 seed 42 사후 행동 진단

## tl;dr

이미 평가한 M1과 `m3_clv_influence` 체크포인트에서 Top-50을 복원해, 실제 신규상품 정답·추천상품·사용자 성향·상품 승격 메커니즘을 상세히 비교합니다. 이 노트북은 학습이나 체크포인트 선택을 하지 않습니다. 실행 결과를 이용해 같은 test 구간에 맞춘 모형을 다시 확증해서는 안 됩니다.

## Context & Methods

- 대상: 학습구간에 없고 DAY 698–704에 등장한 신규 사용자–상품 정답
- 비교: 동일 seed의 `m1_baseline` 대 `m3_clv_influence`
- 사용자 성향: 학습구간의 거래횟수 N, 평균 거래금액 V, N×V 백분위만 사용
- 산출물: 사용자별 정답/Top-50, 성향별 지표, 상품 특성, 정답 순위 이동, 상품별 CLV 이웃 구성, 대표 사례, QA

### Key Assumptions

저장된 원천 데이터·실행 JSON·체크포인트 해시가 기존 실행과 같아야 합니다. 한 seed 결과이므로 분산·신뢰구간·유의성을 주장하지 않습니다. 가격·구매금액 가중 적중값은 실제 증분매출이 아닙니다.

## Data

### 1. Drive와 고정 코드 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess, urllib.request

DIAGNOSTIC_SHA = '7e361cfda52458653ec431e345e6e6750f97e441'
repo = Path('/content/clv-m2-lightgcn-runner')
archive = Path('/content/m3-diagnostic-source.zip')
extracted = Path(f'/content/clv-m2-lightgcn-runner-{DIAGNOSTIC_SHA}')
for path in (repo, extracted):
    if path.exists():
        shutil.rmtree(path)
urllib.request.urlretrieve(
    f'https://github.com/jung-un/clv-m2-lightgcn-runner/archive/{DIAGNOSTIC_SHA}.zip',
    archive,
)
shutil.unpack_archive(archive, '/content')
extracted.rename(repo)
%cd /content/clv-m2-lightgcn-runner
print('Pinned diagnostic source:', DIAGNOSTIC_SHA)

### 2. 사전 조건 확인

In [ ]:
import json
from lightgcn_clv_m3_behavior_diagnostic import (
    configure_m3_behavior_diagnostic,
    preflight_summary,
    run_m3_behavior_diagnostic,
)

cfg = configure_m3_behavior_diagnostic(seeds=(42,))
preflight = preflight_summary(cfg)
assert preflight['training'] is False
assert preflight['checkpoint_selection'] is False
assert preflight['seeds'] == [42]
print(json.dumps(preflight, ensure_ascii=False, indent=2))

## Results

### 3. 체크포인트 복원 및 전체 진단 실행

In [ ]:
segment_metrics = run_m3_behavior_diagnostic(cfg)
result_paths = segment_metrics.attrs['result_paths']
assert segment_metrics.attrs['quality_passed'] is True
print(json.dumps(result_paths, ensure_ascii=False, indent=2))

### 4. QA와 전체 사용자 지표 재현

In [ ]:
import pandas as pd
from IPython.display import display

quality = pd.read_csv(result_paths['quality_checks'])
display(quality)
assert quality['passed'].all(), quality.loc[~quality['passed']]
user_metrics = pd.read_csv(result_paths['user_metrics'])
display(user_metrics.groupby('model_id')[['recall@10', 'ndcg@10', 'recall@50', 'ndcg@50']].mean())

### 5. N/V 성향 및 CLV 5분위별 성과

In [ ]:
focus_metrics = ['recall@10', 'ndcg@10', 'recall@50', 'ndcg@50', 'price_purchase_amount_weighted_hit@10']
focus = segment_metrics[segment_metrics['metric'].isin(focus_metrics)].copy()
display(focus.sort_values(['segment_type', 'segment_id', 'metric']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_data = focus[focus['metric'].isin(['recall@10', 'ndcg@10', 'recall@50'])]
grid = sns.catplot(
    data=plot_data, x='segment_id', y='relative_change_pct',
    hue='metric', col='segment_type', kind='bar', sharex=False, height=4, aspect=1.35
)
grid.set_axis_labels('학습이력 사용자 성향', 'M3 대 M1 변화율 (%)')
grid.set_titles('{col_name}')
for axis in grid.axes.flat:
    axis.axhline(0, color='black', linewidth=0.8)
    axis.tick_params(axis='x', rotation=35)
plt.show()

### 6. 실제 정답 상품이 어느 순위로 이동했는가

In [ ]:
truth = pd.read_csv(result_paths['test_truth_rank_transition'])
transition_summary = truth.groupby(['nv_quadrant', 'clv_quintile']).agg(
    n_truth_items=('item_idx', 'size'),
    mean_rank_improvement=('rank_improvement', 'mean'),
    entered_top10=('entered_top10', 'sum'),
    left_top10=('left_top10', 'sum'),
    entered_top50=('entered_top50', 'sum'),
    left_top50=('left_top50', 'sum'),
).reset_index()
display(transition_summary)
display(truth.sort_values('rank_improvement', ascending=False).head(20))
display(truth.sort_values('rank_improvement').head(20))

### 7. 성향별 정답·M1 추천·M3 추천 상품 특성

In [ ]:
item_profiles = pd.read_csv(result_paths['segment_item_profile'])
profile_columns = [
    'segment_type', 'segment_id', 'role', 'n_occurrences', 'n_unique_items',
    'mean_price_percentile', 'mean_train_user_count', 'mean_repeat_purchase_share',
    'top_category', 'top_category_share',
]
display(item_profiles[profile_columns].sort_values(['segment_type', 'segment_id', 'role']))
category_profiles = pd.read_csv(result_paths['segment_category_profile'])
display(category_profiles.sort_values('occurrence_share', ascending=False).groupby(['segment_type', 'segment_id', 'role']).head(5))

### 8. M3가 승격한 상품과 CLV 이웃 메커니즘

In [ ]:
promoted = pd.read_csv(result_paths['promoted_item_examples'])
correlations = pd.read_csv(result_paths['item_mechanism_correlations'])
display(promoted[[
    'item_id', 'category', 'top10_promotion_count', 'promoted_hit_count',
    'promoted_hit_rate', 'train_neighbor_count', 'mean_train_neighbor_q_clv',
    'high_clv_neighbor_share', 'operator_l1_shift', 'price_percentile',
]].head(50))
display(correlations.sort_values(['target', 'spearman'], ascending=[True, False]))

In [ ]:
mechanism = pd.read_csv(result_paths['item_clv_mechanism'])
sns.scatterplot(
    data=mechanism, x='mean_train_neighbor_q_clv', y='top10_promotion_count',
    size='train_neighbor_count', hue='promoted_hit_count', sizes=(10, 180), alpha=0.65
)
plt.xlabel('상품 학습 구매자의 평균 CLV 백분위')
plt.ylabel('M3에서 Top-10에 새로 들어온 횟수')
plt.title('상품 승격이 고CLV 구매자 구성과 함께 움직이는가')
plt.show()

### 9. 사전 규칙으로 선정한 대표 사용자 사례

In [ ]:
cases = pd.read_csv(result_paths['representative_user_cases'])
case_details = pd.read_csv(result_paths['representative_case_details'])
display(cases.sort_values('selection_rule'))

for _, case in cases.iterrows():
    print('\n', '=' * 90)
    print(case['selection_rule'], '| user:', case['user_id'], '|', case['nv_quadrant_label'], '|', case['clv_quintile'])
    detail = case_details[(case_details['seed'] == case['seed']) & (case_details['user_idx'] == case['user_idx']) & (case_details['selection_rule'] == case['selection_rule'])]
    columns = [column for column in ['detail_role', 'model_id', 'rank', 'item_id', 'category', 'is_test_truth', 'test_purchase_amount', 'price_percentile', 'train_user_count'] if column in detail]
    display(detail[columns].sort_values(['detail_role', 'rank'], na_position='last'))

## Takeaways

다음 모형 방향은 위 결과를 본 뒤 아래 순서로 판단합니다.

1. Top-10 이탈이 Top-50 진입보다 많으면 정답 부재보다 상위 순위 분별 문제를 우선 검토합니다.
2. 승격 횟수가 상품 구매자의 평균 CLV나 상품 인기도와 강하게 연결되면서 승격 적중률이 낮으면, 사용자 맞춤화보다 전역 상품 확산이 일어난 것으로 해석합니다.
3. 특정 N/V 집단에서만 일관된 개선이 보이면 CLV 전체 곱의 보편적 효과가 아니라 집단 이질성 가설로 제한합니다.
4. 어떤 후속 구조도 이 test에 맞춰 재평가하지 않고 새 사전 고정 시간분할 또는 독립 데이터에서 검증합니다.

실행 후 `segment_metric_summary`, `test_truth_rank_transition`, `item_mechanism_correlations`, `representative_user_cases` 표를 공유하면 다음 구조의 개선 가설을 근거와 함께 좁힐 수 있습니다.